# Lab 8: Decisions, experiments, Bayes learning and bandits

AII 600. **Due Wednesday 18 November 2026, before class. Submit on Canvas.**

**How to submit.** Fill in every exercise cell. Choose Kernel, then Restart and Run All. When every check prints `ok`, upload the `.ipynb` file on Canvas.

**How the checks work.** Each exercise ends with a check cell.

- `ok  name`: your value matches the reference.
- `not yet: ...` (an error): it does not match, or is still `None`. Re-read the task, the inputs and the rounding, then run again.
- Exercises with a simulation (marked MC) accept a small tolerance, stated in the task. Use your own `np.random.default_rng(seed)`.
- Run All stops at the first empty exercise. That is expected. Work in order.

Each exercise defines the variables it names. Do not rename them. Do not edit the check cells.

| # | Topic | Defines | Slides |
|---|---|---|---|
| 1 | Utility: St. Petersburg certainty equivalent | `ce_log` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 2 | Utility: St. Petersburg with a finite bankroll | `fair_price_1m` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 3 | Kelly: half-Kelly stake | `stake_half` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 4 | Winner's curse: break-even bid shading | `k_even` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 5 | Value of a test | `prior_lo`, `prior_hi` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 6 | Nash: mixed equilibrium | `pA`, `pB` | [Decisions](https://vsokolov.org/courses/files/600/lab8-decisions.html) |
| 7 | A/B test: interval for a difference of rates | `diff_lo`, `diff_hi` | formula in the task |
| 8 | A/B test: sample size per arm | `n_arm` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html), power slide |
| 9 | Beta posterior, mean and 95% interval (SimCity) | `post_a`, `post_b`, `post_mean`, `ci_lo`, `ci_hi` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 10 | Posterior predictive for black swans (MC) | `al_n`, `be_n`, `pred_white`, `p_black100` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 11 | Gamma posterior for an event rate | `a_t`, `A_t`, `lam_mean`, `lam_lo`, `lam_hi` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 12 | Bears shrinkage and predictive probability | `B`, `mu_n`, `post_sd`, `p_beat_line` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 13 | Which arm is better (MC) | `p_3_over_2`, `p_3_best` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 14 | Thompson sampling with a fixed seed (MC) | `share_best` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |
| 15 | Q <- Q + alpha (r - Q), alpha = 1/n against constant alpha | `q_1n`, `q_const` | [Bayes and bandits](https://vsokolov.org/courses/files/600/bayes-bandits.html) |

In [ ]:
from hashlib import sha256

import numpy as np
from scipy import stats

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    try:
        s = f"{round(float(value), ndigits):.{ndigits}f}"
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a single number, not {type(value).__name__}")
    if not np.isfinite(float(value)):
        raise AssertionError(f"not yet: {name} is not finite (nan or inf)")
    got = sha256(f"aii600-bayes-bandits|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} does not match the reference value (you got {s}); check the formula and the inputs"
    print("ok ", name)

def check_array(name, value, expected, shape, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    value = np.asarray(value, dtype=float)
    if value.shape != shape:
        raise AssertionError(f"not yet: {name} must have shape {shape}, but has shape {value.shape}")
    if not np.all(np.isfinite(value)):
        raise AssertionError(f"not yet: {name} contains nan or inf")
    s = ",".join(f"{round(float(v), ndigits):.{ndigits}f}" for v in value.ravel())
    got = sha256(f"aii600-bayes-bandits|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} has the right shape but the values do not match the reference; check each update"
    print("ok ", name)

def check_close(name, value, accepted, width):
    # simulation check: passes when the value is within about `width` of the reference
    if value is None:
        raise AssertionError(f"not yet: replace None for {name}")
    try:
        v = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} must be a single number, not {type(value).__name__}")
    if not np.isfinite(v):
        raise AssertionError(f"not yet: {name} is not finite (nan or inf)")
    k = int(round(v / width))
    got = sha256(f"aii600-bayes-bandits|{name}|bin{k}".encode()).hexdigest()[:16]
    assert got in accepted, f"not yet: {name} is not within {width} of the reference (you got {v:.4f}); check the formula, the inputs and the number of draws"
    print("ok ", name)

import pandas as pd

def check_t(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"not yet: {name} is still None. Fill in the exercise cell above and run it.")
    try:
        x = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"not yet: {name} should be one number, but it is {type(value).__name__}.")
    if not np.isfinite(x):
        raise AssertionError(f"not yet: {name} is not a finite number ({x}).")
    s = f"{round(x, ndigits):.{ndigits}f}"
    got = sha256(f"aii600-takehome|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"not yet: {name} = {x:.6g} does not match. Check the formula and the inputs."
    print("ok ", name)


## Part 1: Decisions and A/B tests

### Exercise 1: St. Petersburg, log utility

A game pays $2^T$ dollars when the first head is on toss $T$, with chance $2^{-T}$. Utility is $u(x) = \log x$ (natural log).

Inputs: $T = 1, 2, \ldots, 200$.

Formulas:

$$
\text{EU} = \sum_{T=1}^{200} 2^{-T} \log\left(2^{T}\right), \qquad \text{CE} = e^{\text{EU}}.
$$

Define `ce_log`: CE, in dollars. One number, not rounded.


In [ ]:
# Exercise 1: define ce_log
ce_log = None


In [ ]:
check_t("ce_log", ce_log, "cb9adfc8e2f1cc6c")
print("ok  Exercise 1")


### Exercise 2: St. Petersburg with a finite bankroll

The game of Exercise 1 is offered by a casino with a bankroll of \$1,000,000. When $2^T$ would be above \$1,000,000, it pays \$1,000,000.

Inputs: $T = 1, 2, \ldots, 200$.

Formula, the fair price:

$$
\text{price} = \sum_{T=1}^{200} 2^{-T} \min\left(2^{T},\, 1{,}000{,}000\right).
$$

Define `fair_price_1m`: the fair price, in dollars. One number, not rounded.


In [ ]:
# Exercise 2: define fair_price_1m
fair_price_1m = None


In [ ]:
check_t("fair_price_1m", fair_price_1m, "8276e7b47dea71db")
print("ok  Exercise 2")


### Exercise 3: Kelly stake

A bet has payout $O = 5$: a winning \$1 ticket pays back \$5, the \$1 included. Your chance of winning is $c = 0.28$. Your bankroll is $B = \$8{,}000$.

Formulas:

$$
\text{edge} = cO - 1, \qquad f = \frac{cO - 1}{O - 1}, \qquad \text{stake} = \tfrac{1}{2}\, f\, B.
$$

Here $f$ is the Kelly fraction of the bankroll. The stake uses half of it.

Define `stake_half`: the dollar stake at half Kelly. One number, not rounded.


In [ ]:
# Exercise 3: define stake_half
stake_half = None


In [ ]:
check_t("stake_half", stake_half, "a8423a775fa8d743")
print("ok  Exercise 3")


### Exercise 4: winner's curse

Five bidders bid on one item. The cell below simulates 100,000 auctions (seed 5): `value5` is the true value $V$ in each auction, shape `(100000,)`. `signals5` holds the five noisy estimates, shape `(100000, 5)`.

Each bidder bids $k$ times his own estimate. The winner is the bidder with the highest estimate $M$, and he pays his own bid $kM$. His profit in one auction is $V - kM$.

Task: find $k$ at which the average profit over the 100,000 auctions is 0. Solve $\text{mean}(V - kM) = 0$ for $k$.

Define `k_even`: one number, about 0.9, not rounded. The check compares 3 decimals.


In [ ]:
def auctions(n_bidders, n_auctions=100_000, seed=5):
    rng = np.random.default_rng(seed)
    value = rng.normal(100, 10, n_auctions)                                # true value, dollars
    signals = value[:, None] + rng.normal(0, 10, (n_auctions, n_bidders))  # noisy estimates
    return value, signals

value5, signals5 = auctions(5)
print(value5.shape, signals5.shape)


In [ ]:
# Exercise 4: define k_even
k_even = None


In [ ]:
check_t("k_even", k_even, "e8eb0b91e9884936", ndigits=3)
print("ok  Exercise 4")


### Exercise 5: value of a test

A disease has prior probability `prior`. A test has sensitivity 0.95 and specificity 0.99. Utilities, in points: treat gives 90 (healthy or sick). Do not treat gives 100 if healthy and 0 if sick. The cell below defines `priors` (0.000, 0.001, ..., 1.000) and `value_of_test(prior)`: the expected utility of testing and then choosing the better action, minus the best expected utility without the test.

The test costs 1 point: `test_cost = 1`.

Define `prior_lo` and `prior_hi`: the smallest and the largest value in `priors` for which `value_of_test(prior)` is above `test_cost`. Each is one number between 0 and 1.


In [ ]:
sens, spec = 0.95, 0.99
test_cost = 1                                   # points
priors = np.arange(1001) / 1000                 # 0.000, 0.001, ..., 1.000

def eu_best(p_sick):
    return max(90, 100 * (1 - p_sick))          # treat: 90 for sure; do not treat: 100 if healthy

def value_of_test(prior):
    p_pos = sens * prior + (1 - spec) * (1 - prior)
    p_neg = 1 - p_pos
    post_pos = sens * prior / p_pos             # chance of disease after a positive test
    post_neg = (1 - sens) * prior / p_neg       # chance of disease after a negative test
    return p_pos * eu_best(post_pos) + p_neg * eu_best(post_neg) - eu_best(prior)

print(round(value_of_test(0.02), 4))


In [ ]:
# Exercise 5: define prior_lo, prior_hi
prior_lo = None
prior_hi = None


In [ ]:
check_t("prior_lo", prior_lo, "6f3d0d549e585eec")
check_t("prior_hi", prior_hi, "8e81cb41bc98c4f4")
print("ok  Exercise 5")


### Exercise 6: marble game

Players $A$ and $B$ each show a red or a blue marble at the same time. Payoffs, in dollars, paid by an outside organizer; the other player gets \$0:

| | $B$ red | $B$ blue |
|---|---|---|
| $A$ red | $A$ wins 4 | $B$ wins 2 |
| $A$ blue | $B$ wins 2 | $A$ wins 1 |

Let `pA` be the chance that $A$ shows red and `pB` the chance that $B$ shows red. At the mixed equilibrium each player is indifferent between his two marbles:

- $B$ shows red with chance $p_B$: $A$'s expected payoff from red equals that from blue.
- $A$ shows red with chance $p_A$: $B$'s expected payoff from red equals that from blue.

Define `pA` and `pB`: each one number between 0 and 1, exact, with a fraction written as a decimal (1/8 is 0.125).


In [ ]:
# Exercise 6: define pA, pB
pA = None
pB = None


In [ ]:
check_t("pA", pA, "1710a6c7acd62fcc")
check_t("pB", pB, "fc03f3492ae0ca7f")
print("ok  Exercise 6")


### Exercise 7: interval for a difference of rates

An A/B test of two banners. Pink: 200 purchases among 500 visitors. Black: 165 purchases among 550 visitors.

Inputs: `n_pink`, `buys_pink`, `n_black`, `buys_black` (cell below).

Formulas, with $\hat p_1 = \text{buys\_pink}/\text{n\_pink}$ and $\hat p_2 = \text{buys\_black}/\text{n\_black}$:

$$
\text{SE} = \sqrt{\frac{\hat p_1 (1 - \hat p_1)}{n_1} + \frac{\hat p_2 (1 - \hat p_2)}{n_2}}, \qquad
(\hat p_1 - \hat p_2) \pm z \cdot \text{SE}, \quad z = 1.96.
$$

Use `stats.norm.ppf(0.975)` for $z$.

Define `diff_lo` and `diff_hi`: the lower and upper ends of the 95% interval for pink minus black. Two numbers, as rates (0.04 means 4 points), not rounded.


In [ ]:
n_pink, buys_pink = 500, 200
n_black, buys_black = 550, 165
print(buys_pink / n_pink, buys_black / n_black)


In [ ]:
# Exercise 7: define diff_lo, diff_hi
diff_lo = None
diff_hi = None


In [ ]:
check_t("diff_lo", diff_lo, "1c56e9f16a5d0ca3")
check_t("diff_hi", diff_hi, "24e67267fae872fe")
print("ok  Exercise 7")


### Exercise 8: sample size per arm

Task: the number of visitors per arm needed to detect a lift in the purchase rate from $p_{\text{old}} = 0.30$ to $p_{\text{new}} = 0.32$, with 5% significance (two-sided) and 80% power.

Formula:

$$
n = \frac{(z_{0.975} + z_{0.80})^2 \left[p_{\text{old}}(1 - p_{\text{old}}) + p_{\text{new}}(1 - p_{\text{new}})\right]}{(p_{\text{new}} - p_{\text{old}})^2}.
$$

Here $z_{0.975}$ and $z_{0.80}$ are normal quantiles. Use `stats.norm.ppf(0.975)` and `stats.norm.ppf(0.80)`, not 1.96 and 0.84. Round up with `np.ceil` and convert to `int`.

Define `n_arm`: visitors per arm, an integer.


In [ ]:
# Exercise 8: define n_arm
n_arm = None


In [ ]:
check_t("n_arm", n_arm, "9f17504e476314a4")
print("ok  Exercise 8")


## Part 2: Bayes learning and bandits

### Exercise 9: Beta posterior (SimCity)

Task: find the posterior of the pink page's purchase rate from a flat prior, its mean, and its 95% equal-tailed credible interval.

Inputs: `buys_pink = 200` buys among `n_pink = 500` visitors (book SimCity data). Prior Beta(`a0`, `b0`) = Beta(1, 1).

Formulas: the posterior is Beta(`post_a`, `post_b`) with `post_a = a0 + buys_pink` and `post_b = b0 + n_pink - buys_pink`. The mean is `post_a / (post_a + post_b)`. The interval is the 2.5% and 97.5% quantiles of the posterior (`stats.beta.ppf`).

Define: `post_a`, `post_b`, `post_mean`, `ci_lo`, `ci_hi`. Rates as numbers between 0 and 1, no rounding.


In [ ]:
buys_pink, n_pink = 200, 500
a0, b0 = 1, 1


In [ ]:
# Exercise 9: define post_a, post_b, post_mean, ci_lo, ci_hi


In [ ]:
check("post_a", post_a, "a5dd147257132ffd")
check("post_b", post_b, "aa703b16d6fe1d26")
check("post_mean", post_mean, "20d57acff0c1d51b")
check("ci_lo", ci_lo, "7d43c2af41351347")
check("ci_hi", ci_hi, "fb3eebfa7a45a259")
print("ok  Exercise 9")


### Exercise 10: Posterior predictive, black swans (MC)

Task: after 100 white swans and no black swan, find the probability that the next swan is white, and the probability of at least one black swan among the next 100.

Inputs: `T_swans = 100` swans seen, `Y_white = 100` of them white. Prior Beta(1, 1).

Formulas: the posterior is Beta(`al_n`, `be_n`) with `al_n = 1 + Y_white` and `be_n = 1 + T_swans - Y_white`. The next swan is white with probability `pred_white = al_n / (al_n + be_n)`. For the next 100 swans, `p_black100 = E[1 - theta^100]` under the posterior. Estimate it by Monte Carlo: draw at least 100,000 values of `theta` from the posterior and average `1 - theta**100`.

Define: `al_n`, `be_n`, `pred_white` (exact formula), `p_black100` (MC, tolerance 0.01).


In [ ]:
T_swans, Y_white = 100, 100


In [ ]:
# Exercise 10: define al_n, be_n, pred_white, p_black100


In [ ]:
check("al_n", al_n, "2d705419153696b6")
check("be_n", be_n, "45b63b77232a7467")
check("pred_white", pred_white, "a1162067d855a3bc")
check_close("p_black100", p_black100, ['1524e3e854e46538', '08c2ba4dbb636b16', '8419622a66ed4de7'], 0.01)
print("ok  Exercise 10")


### Exercise 11: Gamma posterior for an event rate

Task: update a Gamma prior for an event rate with counts, and report the posterior mean and a 90% interval.

Inputs: events per year, rate `lam`. Prior Gamma(`a_prior`, `A_prior`) = Gamma(4, 1), the high-activity prior of the book (mean 4 events per year, 5% to 95% prior range 1.37 to 7.75). Data: `N_events = 4` events observed over `t_years = 4` years.

Formulas: the posterior is Gamma(`a_t`, `A_t`) with `a_t = a_prior + N_events` (shape) and `A_t = A_prior + t_years` (rate). The mean is `a_t / A_t`. The interval is the 5% and 95% quantiles, `stats.gamma.ppf(q, a_t, scale=1 / A_t)`. Check your call on the prior first: it should return 1.37 and 7.75.

Define: `a_t`, `A_t`, `lam_mean`, `lam_lo`, `lam_hi`, in events per year.


In [ ]:
a_prior, A_prior = 4, 1
N_events, t_years = 4, 4


In [ ]:
# Exercise 11: define a_t, A_t, lam_mean, lam_lo, lam_hi


In [ ]:
check("a_t", a_t, "cdda66c9278c0b1b")
check("A_t", A_t, "62610c4656fba083")
check("lam_mean", lam_mean, "d237b3cae25b942a")
check("lam_lo", lam_lo, "0589a3d66d0f95ff")
check("lam_hi", lam_hi, "6d0b95b94e948054")
print("ok  Exercise 11")


### Exercise 12: Bears shrinkage and predictive probability

Task: shrink the Bears' average margin toward an average team, and find the probability that the next game beats the betting line.

Inputs: `margins` (points, own score minus opponent score, 9 games, book data), prior mean `mu0 = 0` and prior standard deviation `tau = 4` points, betting line `line = -3.5` points. Treat the standard deviation of one margin as known: `sigma = s`, the sample standard deviation of `margins` (`ddof=1`).

Formulas, with `n_games = 9` and `ybar` the sample mean:

- `B = tau**2 / (tau**2 + sigma**2 / n_games)`, the weight on the data.
- `mu_n = mu0 + B * (ybar - mu0)`, the posterior mean.
- `post_sd = sqrt(1 / (1 / tau**2 + n_games / sigma**2))`, the posterior standard deviation of the average margin.
- Predictive: the next margin is normal with mean `mu_n` and standard deviation `sqrt(post_sd**2 + sigma**2)`. `p_beat_line` is the probability that it is above `line`.

Define: `B`, `mu_n`, `post_sd`, `p_beat_line`. Points for `mu_n` and `post_sd`, a probability for `p_beat_line`.


In [ ]:
margins = np.array([-3, 8, 8, -21, -7, 14, -13, -28, -41])
mu0, tau, line = 0.0, 4.0, -3.5


In [ ]:
# Exercise 12: define B, mu_n, post_sd, p_beat_line


In [ ]:
check("B", B, "2ed8ac774c26eed8")
check("mu_n", mu_n, "5982280dad08560b")
check("post_sd", post_sd, "d98a47669d68a7ec")
check("p_beat_line", p_beat_line, "fe9c99a748e1c211")
print("ok  Exercise 12")


### Exercise 13: Which arm is better (MC)

Task: from three arms' counts, find the probability that arm 3 beats arm 2, and the probability that arm 3 is the best of the three.

Inputs (book three-arm example): `sessions = [20, 30, 40]`, `conversions = [12, 20, 30]`. Prior Beta(1, 1) for each arm.

Formulas: arm `k` has posterior Beta(`1 + conversions[k]`, `1 + sessions[k] - conversions[k]`). Draw at least 100,000 values from each posterior with your own `np.random.default_rng(seed)`. `p_3_over_2` is the share of draws with arm 3 above arm 2. `p_3_best` is the share of draws with arm 3 above both of the others.

Define: `p_3_over_2`, `p_3_best`, probabilities between 0 and 1 (MC, tolerance 0.01).


In [ ]:
sessions = np.array([20, 30, 40])
conversions = np.array([12, 20, 30])


In [ ]:
# Exercise 13: define p_3_over_2, p_3_best


In [ ]:
check_close("p_3_over_2", p_3_over_2, ['5d4b8336a8105390', 'b01131d8e11f613d', 'db05d3d6843371a6'], 0.01)
check_close("p_3_best", p_3_best, ['dd176faf6654eeeb', '25a82a8551271556', 'df17014310545055'], 0.01)
print("ok  Exercise 13")


### Exercise 14: Thompson sampling with a fixed seed (MC)

Task: run Thompson sampling on two pages and find the share of visitors shown the better page.

Inputs (book bandit example): `rates = [0.04, 0.05]` are the true conversion rates of arm 0 (original) and arm 1 (variation, the better arm). The algorithm does not know them. Run `R = 200` independent experiments of `T = 3000` visitors each, in parallel as arrays of shape `(R, 2)`.

Algorithm, for each of the `T` visitors:

1. Draw one value per arm and experiment from its Beta posterior, `rng.beta(a, b)`. Start with `a` and `b` equal to ones.
2. Show the arm with the larger draw (`argmax` over axis 1).
3. Draw the reward: a conversion with probability `rates[arm]` (`rng.random(R) < rates[arm]`).
4. Add the reward to `a` of the arm shown and `1 - reward` to its `b`.

Use your own `rng = np.random.default_rng(seed)`.

Define: `share_best`, the share of all `R * T` visitors who were shown arm 1. A 50/50 split gives 0.5. Tolerance 0.06.


In [ ]:
rates = np.array([0.04, 0.05])
R, T = 200, 3000


In [ ]:
# Exercise 14: define share_best


In [ ]:
check_close("share_best", share_best, ['71a00ce09552f3ba', '54604e0c49b365ae', '1fdf34fb18728e71'], 0.06)
print("ok  Exercise 14")


### Exercise 15: The incremental update, 1/n against constant alpha

Task: write the update `Q <- Q + alpha * (r - Q)` in a loop over logged visitors and compare two step sizes.

Inputs: `arm_log` (0 = pink, 1 = black) and `reward_log` (1 = buy, 0 = no buy) for 40 simulated visitors, from the cell above. Pink's true rate is 0.6 for the first 20 visitors and 0.2 after that. Keep one estimate per page, both starting at 0.

Formulas: `q_1n` uses `alpha = 1 / n`, where `n` counts the visitors shown that page so far, including this one. `q_const` uses the constant `alpha = 0.2`. For each visitor, update only the estimate of the page shown.

Define: `q_1n` and `q_const`, arrays of shape `(2,)` with the final estimates for (pink, black). No rounding.


In [ ]:
# Simulated log: 40 visitors, arm 0 = pink, arm 1 = black.
rng_q = np.random.default_rng(55)
T_q = 40
arm_log = rng_q.integers(0, 2, T_q)
rate_now = np.where(np.arange(T_q) < 20, 0.6, 0.2)
reward_log = (rng_q.random(T_q) < np.where(arm_log == 0, rate_now, 0.4)).astype(float)
print("visitors:", T_q, " shown pink:", int((arm_log == 0).sum()), " shown black:", int((arm_log == 1).sum()))


In [ ]:
# Exercise 15: define q_1n, q_const


In [ ]:
check_array("q_1n", q_1n, "4b509e7cfa2a6b2b", (2,))
check_array("q_const", q_const, "38833069b5a1259f", (2,))
print("ok  Exercise 15")


## Submitting

Choose Kernel, then Restart and Run All. Every check cell must print `ok`. Upload the `.ipynb` file on Canvas.